# 05 · Exportar resultados a Google Sheets

Lee las tablas del notebook 04 y las escribe en dos hojas del Drive personal, con una cuenta de servicio de Google.

| Hoja | Acceso | Contenido |
|---|---|---|
| `ieb_resultados_publicos` | cualquiera con el enlace (lector) | `leeme`, `resumen`, `por_documento`: cifras, sin texto de facturas |
| `ieb_detalle_restringido` | solo el autor | `detalle`: valor anotado vs. extraído (licencia DocILE: solo investigación) |

**Credencial:** el JSON de la cuenta `ieb-sheets-writer@invoice-extraction-benchmark.iam.gserviceaccount.com` está en el secret
`invoice-extraction-benchmark / gsheets_sa_json`. No aparece en el código ni en el repositorio.

In [ ]:
%pip install -q gspread==6.1.4

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Celda 1: parámetros y conexión
import json, math
import gspread
dbutils.widgets.text("hoja_publica", "1ol8jgRnQ9HZgfhB_4J-Iwp6cpdBMv7odHrgCKMRk4dc")
dbutils.widgets.text("hoja_restringida", "1EKJngJ5CcD_wFrw0g2W6msIYxlyW3VckRtC3TkBe7pg")
gc = gspread.service_account_from_dict(json.loads(dbutils.secrets.get("invoice-extraction-benchmark", "gsheets_sa_json")))
pub = gc.open_by_key(dbutils.widgets.get("hoja_publica"))
res = gc.open_by_key(dbutils.widgets.get("hoja_restringida"))
print("conectado:", pub.title, "|", res.title)

In [ ]:
# Celda 2: escribir
def limpio(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ""
    return v

def escribir(libro, nombre, df):
    filas = [df.columns] + [[limpio(v) for v in r] for r in df.toPandas().itertuples(index=False)]
    try:
        ws = libro.worksheet(nombre)
    except gspread.WorksheetNotFound:
        ws = libro.add_worksheet(nombre, rows=len(filas) + 10, cols=len(df.columns))
    ws.clear()
    ws.resize(rows=len(filas), cols=len(df.columns))
    ws.update(values=filas, range_name="A1", value_input_option="RAW")
    ws.freeze(rows=1)
    print(f"{libro.title} / {nombre}: {len(filas) - 1} filas")

resumen = spark.table("workspace.invoice_extraction_benchmark.ieb_eval_resumen")
from datetime import datetime, timezone
corrida = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
leeme = spark.createDataFrame([
    ("Proyecto", "Extracción de datos de facturas: M1 (LayoutLMv3, predicciones publicadas de DocILE) vs M2 v2 (ai_parse_document + ai_extract, Databricks)"),
    ("Datos", "DocILE (Šimsa et al., ICDAR 2023), partición val: 500 documentos"),
    ("Métrica", "Por valor: cantidad exacta, costo unitario |Δ|<0.01, importe |Δ|<0.50; emparejamiento húngaro de líneas"),
    ("Pruebas", "Shapiro-Wilk → Wilcoxon / t pareada; McNemar exacto por campo; IC 95 % bootstrap (10 000)"),
    ("Código", "github.com/datavisionary-consulting/databricks/tree/main/invoice-extraction-benchmark"),
    ("Última exportación", corrida),
    ("Idempotencia", "Cada corrida reemplaza tablas y hojas completas; con la misma entrada los resultados son idénticos"),
], "campo string, valor string")

escribir(pub, "leeme", leeme)
escribir(pub, "resumen", resumen)
escribir(pub, "por_documento", spark.table("workspace.invoice_extraction_benchmark.ieb_eval_por_doc"))
escribir(res, "detalle", spark.table("workspace.invoice_extraction_benchmark.ieb_eval_detalle"))
for libro in (pub, res):
    for ws in libro.worksheets():
        if ws.title in ("Hoja 1", "Sheet1") and len(libro.worksheets()) > 1:
            libro.del_worksheet(ws)
print(pub.url); print(res.url)